# Практическая работа 8. Задание 3
Электрическая сеть из 8 подстанций: Краскал, Прим, проверка безопасных ребер

In [1]:
import heapq
import math

# подстанции 1..8 -> индексы 0..7, ребра (стоимость, u, v)
# вес линии 2-6 на рисунке закрыт подписью линии 3-5, взял 5
edges = [
    (7, 0, 1), (4, 0, 2), (9, 0, 3),   # 1-2, 1-3, 1-4
    (9, 1, 2), (3, 1, 4), (5, 1, 5),   # 2-3, 2-5, 2-6
    (5, 2, 3), (2, 2, 4),              # 3-4, 3-5
    (7, 3, 5), (5, 3, 6),              # 4-6, 4-7
    (4, 4, 5), (6, 4, 7),              # 5-6, 5-8
    (3, 5, 6), (5, 5, 7),              # 6-7, 6-8
    (2, 6, 7)                          # 7-8
]
n = 8

adj = {u: [] for u in range(n)}
for w, u, v in edges:
    adj[u].append((w, v))
    adj[v].append((w, u))


class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        # сжатие путей
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        # объединение по рангу
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        return True


def kruskal(n, edges):
    # edges: [(weight, u, v), ...]
    edges = sorted(edges)
    dsu = DSU(n)
    mst = []
    total = 0
    for w, u, v in edges:
        if dsu.union(u, v):
            mst.append((u, v, w))
            total += w
            if len(mst) == n - 1:
                break
    return mst, total


def prim(n, adj, start=0):
    # adj: {u: [(weight, v), ...]}
    visited = [False] * n
    mst = []
    total = 0
    pq = [(0, start, -1)]  # (weight, vertex, parent)
    while pq and len(mst) < n - 1:
        w, u, p = heapq.heappop(pq)
        if visited[u]:
            continue
        visited[u] = True
        if p != -1:
            mst.append((p, u, w))
            total += w
        for weight, v in adj[u]:
            if not visited[v]:
                heapq.heappush(pq, (weight, v, u))
    return mst, total


def cut_for_edge(mst, n, a, b):
    # убираем ребро a-b из дерева, вершины, достижимые из a, это множество S
    tree = {u: [] for u in range(n)}
    for u, v, w in mst:
        if {u, v} != {a, b}:
            tree[u].append(v)
            tree[v].append(u)
    S = {a}
    stack = [a]
    while stack:
        x = stack.pop()
        for y in tree[x]:
            if y not in S:
                S.add(y)
                stack.append(y)
    return S

In [2]:
mst_k, total_k = kruskal(n, edges)
s = ', '.join('%d-%d(%d)' % (u + 1, v + 1, w) for u, v, w in mst_k)
print('Краскал:', s, ' стоимость', total_k)
mst_p, total_p = prim(n, adj, 0)
s = ', '.join('%d-%d(%d)' % (u + 1, v + 1, w) for u, v, w in mst_p)
print('Прим:   ', s, ' стоимость', total_p)
print()

# проверка безопасного ребра для каждого ребра MST
for u, v, w in mst_k:
    S = cut_for_edge(mst_k, n, u, v)
    cross = [e for e in edges if (e[1] in S) != (e[2] in S)]
    m = min(e[0] for e in cross)
    ok = 'безопасное' if w == m else 'нет'
    s = sorted(x + 1 for x in S)
    e = '%d-%d (%d)' % (u + 1, v + 1, w)
    print('ребро %s: S = %s, минимум %d, %s' % (e, s, m, ok))

print()
E, V = len(edges), n
print('E =', E, ' V =', V)
print('Краскал: E*log2(E) =', round(E * math.log2(E), 1))
print('Прим:    E*log2(V) =', round(E * math.log2(V), 1))

Краскал: 3-5(2), 7-8(2), 2-5(3), 6-7(3), 1-3(4), 5-6(4), 3-4(5)  стоимость 23
Прим:    1-3(4), 3-5(2), 5-2(3), 5-6(4), 6-7(3), 7-8(2), 3-4(5)  стоимость 23

ребро 3-5 (2): S = [1, 3, 4], минимум 2, безопасное
ребро 7-8 (2): S = [1, 2, 3, 4, 5, 6, 7], минимум 2, безопасное
ребро 2-5 (3): S = [2], минимум 3, безопасное
ребро 6-7 (3): S = [1, 2, 3, 4, 5, 6], минимум 3, безопасное
ребро 1-3 (4): S = [1], минимум 4, безопасное
ребро 5-6 (4): S = [1, 2, 3, 4, 5], минимум 4, безопасное
ребро 3-4 (5): S = [1, 2, 3, 5, 6, 7, 8], минимум 5, безопасное

E = 15  V = 8
Краскал: E*log2(E) = 58.6
Прим:    E*log2(V) = 45.0
